In [0]:
from pyspark.sql import functions as F
applications = spark.table("fct_applications").filter(F.col('is_hired')==1)
applications = applications.withColumn('hired_date', F.to_date(F.col('hired_date')))
applications = applications.withColumn('apply_date', F.to_date(F.col('apply_date')))
applications = applications.withColumn('time_to_hire_days', F.datediff(F.col('hired_date'), F.col('apply_date'))).filter(F.col('time_to_hire_days')>=0)

job = spark.table('dim_job')
final_df = applications.alias("a").join(job.alias("j"), F.col("a.job_id") == F.col("j.job_id"), how='inner')
final_df = final_df.select('a.job_id', 'j.department', 'a.application_id', 'a.apply_date', 'a.hired_date', 'a.time_to_hire_days')

agg_job = final_df.groupBy('job_id').agg(F.avg('time_to_hire_days').alias('avg_time_to_hire_days'))
agg_dept = final_df.groupBy('department').agg(F.avg('time_to_hire_days').alias('avg_time_to_hire_days'))


# -----------------------------
# Write table
# -----------------------------
final_df.write.format("delta").mode("overwrite").saveAsTable("fct_time_to_hire_detail")
agg_job.write.format("delta").mode("overwrite").saveAsTable("job_time_to_hire")
agg_dept.write.format("delta").mode("overwrite").saveAsTable("department_time_to_hire")

In [0]:
spark.sql('SELECT COUNT(*) FROM fct_time_to_hire_detail;').show()
spark.sql('SELECT * FROM fct_time_to_hire_detail LIMIT 10;').show()
spark.sql('SELECT * FROM job_time_to_hire LIMIT 10;').show()
spark.sql('SELECT * FROM department_time_to_hire LIMIT 10;').show()